# Config

In [1]:
import os
import glob
import xmltodict
import json
import requests
import pandas as pd
import tarfile
import elasticsearch as es
from elasticsearch.helpers import scan
import os
import copy
import random
import datetime
from dotenv import load_dotenv

load_dotenv(dotenv_path="./.varenv")
ELASTIC_URL = os.getenv("ELASTIC_URL")
ELASTIC_USER = os.getenv("ELASTIC_USER")
ELASTIC_PASS = os.getenv("ELASTIC_PASS")

es_server_prod = es.Elasticsearch(
    ELASTIC_URL,
    basic_auth=(ELASTIC_USER, ELASTIC_PASS)
)


es_server_test = es.Elasticsearch(
    "http://localhost:9200"
)


body = json.dumps({
  "mappings": {
    "properties": {
      "ExternalReference" : {
        "type" : "nested"
      }
    }
  }
})

pack_path = "../../../NomenclaturePack25/Orphanet_Nomenclature_Pack_"
langs = ["cs", "de", "en", "es", "fr", "it", "nl", "pl", "pt"]
inactiv_codes = ["513", "8208", "8225", "8449"]

relation = {"cs" : "E (přesné mapování (termíny a pojmy jsou rovnocenné))",
           "de" : "Genaues Mapping (der Begriff und das Konzept ist äquivalent)",
           "en" : "E (Exact mapping: the two concepts are equivalent)",
           "es" : "correspondencia exacta (los términos y los conceptos son equivalentes)",
           "fr" : "E (Alignement exact: les deux concepts sont équivalents)",
           "it" : "mappatura corretta (i termini e i concetti sono equivalenti)",
           "nl" : "exacte overeenkomst (de termen en concepten zijn equivalent)",
           "pl" : "E (dokładne mapowanie (terminy i pojęcia są równoważne)",
           "pt" : "Direção exacta (os termos e os conceitos são equivalentes)"}

validation = {"cs" : "Ověřeno",
           "de" : "Validiert",
           "en" : "Validated",
           "es" : "Validado",
           "fr" : "Validé",
           "it" : "Confermato",
           "nl" : "Gevalideerd",
           "pl" : "Zwalidowany",
           "pt" : "Validado"}

parentPref = {"cs" : "Preferenční rodič",
           "de" : "Bevorzugte Zuordnung",
           "en" : "Preferential parent",
           "es" : "Cabeza de clasificación preferencial",
           "fr" : "Parent préférentiel",
           "it" : "Termine madre preferenziale",
           "nl" : "Preferentiële ouder",
           "pl" : "Uprzywilejowany rodzic",
           "pt" : "Progenitor preferencial"}

In [4]:
es_server_prod.info()

ObjectApiResponse({'name': 'instance-0000000034', 'cluster_name': '9d2d8c7975624d95aa964a1d22a96daf', 'cluster_uuid': '8U1t1cyWQcKEz_Vj8SMaXA', 'version': {'number': '8.19.9', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'f60dd5fdef48c4b6cf97721154cd49b3b4794fb0', 'build_date': '2025-12-16T22:07:42.115850075Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

## Load Snomed mapping file

In [5]:
def load_snomed():
    df = pd.read_excel("ORPHA-SNOMEDCT_Mapping_File_production_2025.xlsx", skiprows=[1, 2])
    snomed = df.astype(str).set_index(df.columns[0])["Unnamed: 2"].to_dict()
    return snomed

snomed = load_snomed()

## Load diff file

Fields:
- orphacode
- preferred term
- classifification level  
- totalstatus
- status update {
    - type string "newly included" "newly inactive"
    - inactive update {
        - type string "deprecated" "obsolete" or "non rare"
        - target orpha or null
        - target preferred term or null 
        - aggregation orpha or null 
        - aggregation preferred term or null } or {} }
- classif update {
  - previous classif
  - new classif } or {} 
- icd10 {
    - type string "new" "update" "remove"
    - new code or null if removed } or {}
- icd11 {
    - type string "new" "update" "remove"
    - new code or null if removed } or {}

In [6]:
prefixes = {
    "Newly included ORPHAcodes" : "newly_included_",
    "Newly Inactive ORPHAcodes" : "newly_inactive_",
    "Classification level update" : "classif_update_",
    "New ORPHA-to-ICD10 mapping" : "new_icd10_",
    "Updated ORPHA-to-ICD10 mapping" : "update_icd10_",
    "Removed ORPHA-to-ICD10 mapping" : "removed_icd10_",
    "New ORPHA-to-ICD11 mapping" : "new_icd11_",
    "Updated ORPHA-to-ICD11 mapping" : "update_icd11_",
    "Removed ORPHA-to-ICD11 mapping" : "removed_icd11_"
}

def diffLoad():
    result = { } 
    
    names = pd.ExcelFile("../../../NomenclaturePack25/Orphanet_Nomenclature_Pack_EN/ORPHAnomenclature_diff_en_2025.xlsx").sheet_names
    df_dict = pd.read_excel("../../../NomenclaturePack25/Orphanet_Nomenclature_Pack_EN/ORPHAnomenclature_diff_en_2025.xlsx", skiprows=1, sheet_name=names)
    
    for sheet, tmp in df_dict.items():
        if "ORPHAcode" not in tmp.columns:
            new_header = tmp.iloc[0] 
            df = tmp[1:] 
            df.columns = new_header
        else:
            df = tmp
        for index, row in df.iterrows():
            orpha = str(row["ORPHAcode"])
            if orpha not in result:
                result[orpha] = { } 
            for key in row.keys():
                result[orpha].update({prefixes[sheet] + key : row[key]})
    return result

diff = diffLoad()

In [7]:
def diffGet(dico, key):
    result = False
    for sheets in prefixes.values():
        result = dico.get(sheets + key, False)
        if result:
            return result

def diffFormat(diff):
    result = { } 
    for k, v in diff.items():
        result[k] = {
            "ORPHAcode" : None,
            "Preferred term" : None,
            "ClassificationLevel" : None,
            "Status": None,
            "Status Update" : { },
            "Classification Update" : { },
            "ICD10 Update" : { },
            "ICD11 Update" : { }
        }
        result[k]["ORPHAcode"] = diffGet(v, "ORPHAcode")
        result[k]["Preferred term"] = diffGet(v, "Name (Preferred_Term)")
        result[k]["ClassificationLevel"] = diffGet(v, "ClassificationLevel")
        
        
        #Status par defaut active sauf feuilles inactive
        
        #diffget a chaque fois et remplir selon résultat ? comment vérifier quon get la bonne feuille ? 
        
        
    return result
    
finalDiff = diffFormat(diff)

# Build main dictionary

## *parcours_prof* : Parses classifications files

In [8]:
def parcours_prof(node, current_classif, final_dict, parent):
    if current_classif["ID of the classification"] == "235":
        node = node["ClassificationNodeChildList"]["ClassificationNode"]
    if "Disorder" in node:
        typology = node["Disorder"]["DisorderType"]["Name"]["#text"]
        orphacode = node["Disorder"]["OrphaCode"]
        children = []
    else:
        return
        
    if node['ClassificationNodeChildList']['@count'] != '0':
        if node['ClassificationNodeChildList']['@count'] == '1':
            children.append(parcours_prof(node['ClassificationNodeChildList']['ClassificationNode'], current_classif, final_dict, orphacode)) 
        else:
            for child in node['ClassificationNodeChildList']['ClassificationNode']:
                children.append(parcours_prof(child, current_classif, final_dict, orphacode)) 
    
    if orphacode in final_dict.keys():
        exists = False
        tmp_classif = final_dict[orphacode].get("Classification", [])
        for classif in tmp_classif:
            if current_classif["ID of the classification"] == classif["ID of the classification"]:
                exists = True
                break
        if not exists:
            tmp_classif.append(current_classif)
    else:
        final_dict[orphacode] = { }
        tmp_classif = [current_classif]

    tmpParent = [parent] if parent else []
    if not "Inheritance" in final_dict[orphacode]:
        final_dict[orphacode]["Inheritance"] = { }
    elif current_classif["ID of the classification"] in final_dict[orphacode]["Inheritance"] :
        tmpParent = final_dict[orphacode]["Inheritance"][current_classif["ID of the classification"]].get("Parent", None)
        if tmpParent:
            tmpParent.append(parent)
    
    final_dict[orphacode]["Inheritance"].update({
        current_classif["ID of the classification"] : {
            "Parent" : tmpParent,
            "Children" : children if children else None
        }
    })
    tmp = { 
        "Typology" : typology,
        "Classification" : tmp_classif,
        "Inheritance" : final_dict[orphacode]["Inheritance"]
    }
    final_dict.update({orphacode: tmp})
            
    return orphacode

## *raw_extraction* : Parses Mapping, Linearisation and Nomenclature files

In [9]:
def raw_extraction(pack_path, encoding):
    final_dict = {}
    for path_file in glob.glob(pack_path, recursive=True):
        if ".xml" in path_file:
            with open(path_file, encoding=encoding) as file:
                xml = xmltodict.parse(file.read())
                xml = xml['JDBOR']['DisorderList']['Disorder']
                for disorder in xml:
                    orpha = disorder['OrphaCode']
                    name = disorder['Name']['#text']
                    
                    status = disorder.get('Totalstatus')
                    if status:
                        status = status["#text"]
                    
                    flag = disorder.get('FlagValue', None)

                    tmpAssociation = [ ] 
                    if "DisorderDisorderAssociationList" in disorder and disorder["DisorderDisorderAssociationList"].get("@count") != "0":
                        if disorder["DisorderDisorderAssociationList"].get("@count") == "1":
                            elt = disorder["DisorderDisorderAssociationList"]["DisorderDisorderAssociation"]
                            tmpAssociation.append({
                                    "TargetDisorder": {
                                        "ORPHAcode": elt["TargetDisorder"].get("OrphaCode", None),
                                        "Preferred term" : elt["TargetDisorder"].get("Name", {}).get("#text")
                                    },
                                    "RootDisorder": { 
                                        "ORPHAcode" : elt["RootDisorder"].get("OrphaCode", None) ,
                                        "Preferred term" : elt["RootDisorder"].get("Name", {}).get("#text")
                                    },
                                    "DisorderDisorderAssociationType" : elt["DisorderDisorderAssociationType"]["Name"]["#text"]
                                })
                        else:
                            for asso in disorder["DisorderDisorderAssociationList"]["DisorderDisorderAssociation"]:
                                tmpAssociation.append({
                                    "TargetDisorder": {
                                        "OrphaCode": elt["TargetDisorder"].get("OrphaCode", None),
                                        "Name" : elt["TargetDisorder"].get("Name", {}).get("#text")
                                    },                                    "RootDisorder": { 
                                        "OrphaCode" : asso["RootDisorder"]["OrphaCode"],
                                        "Name" : asso["RootDisorder"]["Name"]["#text"]
                                    },
                                    "DisorderDisorderAssociationType" : asso["DisorderDisorderAssociationType"]["Name"]["#text"]
                                })
##                          
                    synonyms = disorder.get('SynonymList', None)
                    if synonyms and len(synonyms.keys()) > 1:
                        synonyms = synonyms["Synonym"]
                        if isinstance(synonyms, list):
                            synonyms = [d["#text"] for d in synonyms]
                        else:
                            synonyms = [synonyms["#text"]]
                    else:
                        synonyms = None
                    
                    classif = disorder.get('ClassificationLevel', None)
                    if classif:
                        classif = classif["Name"]["#text"]
                    
                    definition = disorder.get('SummaryInformationList', None)
                    if definition and definition["@count"] == '1': #7204
                        definition = definition.get("SummaryInformation", None)
                        #at least germans can have 0, 1 or more definitions 
                        if definition["TextSectionList"]["@count"] == '0':
                            definition = definition.get("TextAuto", None)
                            if definition:
                                definition = definition["Info"]["#text"]
                        elif definition["TextSectionList"]["@count"] == '1': #6843
                            definition = definition["TextSectionList"]["TextSection"]["Contents"]
                        else: #361
                            definition = definition["TextSectionList"]["TextSection"][0]["Contents"]    
                    else:
                        definition = None

                    aggreg = disorder.get("AggregationLevelSection")
                    if aggreg and aggreg["AggregationLevelList"]["@count"] != "0":
                        #print(aggreg)
                        aggreg = {
                            "AggregationLevel" : [{ 
                                "ORPHAcode" : disorder["AggregationLevelSection"]["AggregationLevelList"]["AggregationLevel"]["OrphaCode"],
                                "Preferred term" : disorder["AggregationLevelSection"]["AggregationLevelList"]["AggregationLevel"]["PreferredTerm"]["#text"],
                                "AggregationLevelStatus" : disorder["AggregationLevelSection"]["AggregationLevelList"]["AggregationLevel"]["AggregationLevelStatus"]
                                }]
                            }
                    
                    ext_ref = disorder.get("ExternalReferenceList", [])
                    if ext_ref:
                        ext_ref = ext_ref["ExternalReference"]
                    refs_array = []
                    if orpha in final_dict:
                        refs_array = final_dict[orpha]["ExternalReference"]
                    if not isinstance(ext_ref, list):
                        ext_ref = [ext_ref]
                    for ref_tmp in ext_ref:
                        reference = ref_tmp["Reference"]
                        source = ref_tmp["Source"]
                        icd11url = ref_tmp.get("DisorderMappingICDRefUrl", None)
                        icd11uri = ref_tmp.get("DisorderMappingICDRefUri", None)
                        ref = { }
                        ref.update({"Reference" : reference})
                        ref.update({"Source" : source})
                        ref.update({"DisorderMappingRelation" : ref_tmp["DisorderMappingRelation"]["Name"]["#text"]})
                        ref.update({"DisorderMappingValidationStatus" : ref_tmp["DisorderMappingValidationStatus"]["Name"]["#text"]})
                        if 'ICD' in source:
                            ref.update({"DisorderMappingICDRelation" : ref_tmp["DisorderMappingICDRelation"]["Name"]["#text"]})
                            #if icd11uri and icd11url:
                            ref.update({"DisorderMappingICDRefUri" : icd11uri,
                                        "DisorderMappingICDRefUrl" : icd11url})
                        refs_array.append(ref)
                        
                    tmp = { }
                    tmp.update({"ORPHAcode" : orpha})
                    tmp.update({"Preferred term" : name})
                    tmp.update({"OrphanetURL" : "https://www.orpha.net/fr/disease/detail/" + orpha})
                    if status:
                        tmp.update({"Status" : status})
                    if flag:
                        tmp.update({"FlagValue" : flag})
                    tmp.update({"Synonym" : synonyms})
                    if classif:
                        tmp.update({"ClassificationLevel" : classif})
                    if definition:
                        tmp.update({"Definition" : definition})
                    if aggreg:
                        tmp.update({"AggregationlevelSection" : aggreg})
                    if tmpAssociation:
                        tmp.update({"DisorderDisorderAssociation": tmpAssociation})
                    tmp.update({"ExternalReference" : refs_array})
                    
                    if orpha in final_dict:
                        final_dict[orpha].update(tmp)
                    else:
                        final_dict[orpha] = tmp
    return final_dict

## *dict_format* : Parse ExternalReference and add datetime

In [10]:
def dict_format(final_dict):
    for orphacode, elt in final_dict.items():
        
        ### date
        elt.update({"Date" : datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")})
        
        ###  externalreference
        if "ExternalReference" not in elt:
            continue
        for ref in elt["ExternalReference"]:
            if "ICD-10" in ref["Source"]:
                tmp = {
                    "Code ICD10" : ref["Reference"],
                    "DisorderMappingRelation" : ref["DisorderMappingRelation"],
                    "DisorderMappingICDRelation" : ref["DisorderMappingICDRelation"],
                    "DisorderMappingValidationStatus" : ref["DisorderMappingValidationStatus"] 
                }
                if "Code ICD10" in elt:
                    elt["Code ICD10"].append(tmp)
                else:
                    elt.update({"Code ICD10" : [tmp]})
            elif "ICD-11" in ref["Source"]:
                tmp = {
                    "Code ICD11" : ref["Reference"],
                    "DisorderMappingRelation" : ref["DisorderMappingRelation"],
                    "DisorderMappingICDRelation" : ref["DisorderMappingICDRelation"],
                    "DisorderMappingValidationStatus" : ref["DisorderMappingValidationStatus"],
                    "DisorderMappingICDRefUrl" : ref["DisorderMappingICDRefUrl"],
                    "DisorderMappingICDRefUri" : ref ["DisorderMappingICDRefUri"]
                }
                if "Code ICD11" in elt:
                    elt["Code ICD11"].append(tmp)
                else:
                    elt.update({"Code ICD11" : [tmp]})
            elif "SNOMED-CT" in ref["Source"]:
                elt.update({
                    "Code SNOMED-CT" : [
                        {
                            "Code SNOMED-CT" : ref["Reference"],
                            "DisorderMappingRelation" : ref["DisorderMappingRelation"],
                            "DisorderMappingValidationStatus" : ref["DisorderMappingValidationStatus"]
                        }
                    ]
                })
            elif "OMIM" in ref["Source"]:
                tmp = {
                    "Code OMIM" : ref["Reference"],
                    "DisorderMappingRelation" : ref["DisorderMappingRelation"],
                    "DisorderMappingValidationStatus" : ref["DisorderMappingValidationStatus"]
                }
                if "Code OMIM" in elt:
                    elt["Code OMIM"].append(tmp)
                else:
                    elt.update({"Code OMIM" : [tmp]})

## *buildData* : Constructs main dictionary from *parcours_prof*, *raw_extraction* and *dict_format*

In [11]:
def buildData(lang):
    encoding = "UTF-8" if lang in ["cs", "pl", "es", "fr", "nl", "pt", "de", "en"] else "ISO-8859-1" 
    typo_classif_dict = { }
    for path_file in glob.glob(f"../../../NomenclaturePack25/Orphanet_Nomenclature_Pack_{lang.upper()}/Classifications/*"):
        with open(path_file, encoding=encoding) as file:
            xml = xmltodict.parse(file.read())
            current_classif = {
                "ID of the classification": xml['JDBOR']["ClassificationList"]["Classification"]["@id"],
                "Name of the classification": xml['JDBOR']["ClassificationList"]["Classification"]["Name"]["#text"],
                "Preferred term": xml['JDBOR']["ClassificationList"]["Classification"]["Name"]["#text"],
                "ORPHAcode": xml['JDBOR']["ClassificationList"]["Classification"]["OrphaNumber"]
            } 
            xml = xml['JDBOR']['ClassificationList']['Classification']['ClassificationNodeRootList']['ClassificationNode']
            parcours_prof(xml, current_classif, typo_classif_dict, None)
    final_dict = raw_extraction(pack_path + lang.upper() + "/*", encoding)
    #
    for k, v in typo_classif_dict.items():
        if k not in final_dict:
            final_dict[k] = { }
        final_dict[k].update({"Typology" : v["Typology"]})
        final_dict[k].update({"Classification" : v["Classification"]})
        final_dict[k].update({"Inheritance" : v["Inheritance"]})
    #
    for k,v in final_dict.items():
        if k in snomed:
            if not "ExternalReference" in final_dict[k]:
                final_dict[k]["ExternalReference"] = []
            final_dict[k]["ExternalReference"].append({"Reference" : snomed[k],
                                   "Source" : "SNOMED-CT",
                                   "DisorderMappingRelation" : relation[lang],
                                   "DisorderMappingValidationStatus" : validation[lang]})    
    dict_format(final_dict)
    return final_dict

In [12]:
Pack = buildData("en")
#print(json.dumps(Pack['166024'], indent=2))
print(len(Pack))

11239


# Creation of indices dictionaries

In [13]:
packorphaIndices = ["ORPHAcode", "Preferred term", "Status", "Synonym", "ClassificationLevel", "Definition", "Typology", "ExternalReference", "Date"]
rdcodeICD10Indices = ["ORPHAcode", "Preferred term", "Synonym", "OrphanetURL", "Date", "Code ICD10"]
rdcodeICD11Indices = ["ORPHAcode", "Preferred term", "Synonym", "OrphanetURL", "Date", "Code ICD11"]
rdcodeOMIMIndices = ["ORPHAcode", "Preferred term", "Synonym", "OrphanetURL", "Code OMIM", "Date"]
rdcodeSNOMEDIndices = ["ORPHAcode", "Preferred term", "Synonym", "OrphanetURL", "Code SNOMED-CT", "Date"]
rdcodeLinearisationIndices = ["ORPHAcode", "Preferred term", "OrphanetURL", "DisorderDisorderAssociation", "Date"] 
rdcodeNomenclatureIndices = ["ORPHAcode", "OrphanetURL", "Preferred term", "FlagValue", "Status", "Synonym", "ClassificationLevel", "Definition", "Typology", "DisorderDisorderAssociation", "AggregationlevelSection", "Date"]
rdcodeClassifIndices = ["ORPHAcode", "Preferred term", "Classification", "Inheritance", "Date"] 

classifList = ["146", "147", "148", "150", "152", "156", "181", "182", "183", "184", "185", "186", "187", "188", "189", 
"193", "194", "195", "196", "197", "198", "199", "200", "201", "202", "203", "204", "205", "209", "212", "216", "231", "233", "235"]

globals()["packorpha_en_product"] = {}

for lang in langs:
    globals()[f"Pack_{lang}"] = buildData(lang)
    
    globals()[f"rdcode_orpha_icd10_mapping_{lang}"] = {}
    globals()[f"rdcode_orpha_icd11_mapping_{lang}"] = {}
    globals()[f"rdcode_orpha_omim_mapping_{lang}"] = {}
    globals()[f"rdcode_orpha_snomed_mapping_{lang}"] = {}
    globals()[f"rdcode_orphalinearisation_{lang}"] = {}
    globals()[f"rdcode_orphanomenclature_{lang}"] = {}
    
    for classif in classifList :
        globals()[f"rdcode_orphaclassification_{classif}_{lang}"] = {}
        
    for (k, v) in globals()[f"Pack_{lang}"].items():
        if lang == "en":
            globals()[f"packorpha_en_product"].update({ k : {x : y for (x, y) in v.items() if x in packorphaIndices}})
            
        globals()[f"rdcode_orpha_icd10_mapping_{lang}"].update({ k : {x : y for (x, y) in v.items() if x in rdcodeICD10Indices}})
        if ("Code ICD" in globals()[f"rdcode_orpha_icd10_mapping_{lang}"][k]): 
            globals()[f"rdcode_orpha_icd10_mapping_{lang}"][k]["Code ICD"] = globals()[f"rdcode_orpha_icd10_mapping_{lang}"][k].pop("Code ICD10")
            
        globals()[f"rdcode_orpha_icd11_mapping_{lang}"].update({ k : {x : y for (x, y) in v.items() if x in rdcodeICD11Indices}})
        if ("Code ICD" in globals()[f"rdcode_orpha_icd10_mapping_{lang}"][k]): 
            globals()[f"rdcode_orpha_icd11_mapping_{lang}"][k]["Code ICD"] = globals()[f"rdcode_orpha_icd11_mapping_{lang}"][k].pop("Code ICD11")
    
#        if all(e in v.keys() for e in rdcodeOMIMIndices):        
        globals()[f"rdcode_orpha_omim_mapping_{lang}"].update({ k : {x : y for (x, y) in v.items() if x in rdcodeOMIMIndices}})

#        if all(e in v.keys() for e in rdcodeSNOMEDIndices):
        globals()[f"rdcode_orpha_snomed_mapping_{lang}"].update({ k : {x : y for (x, y) in v.items() if x in rdcodeSNOMEDIndices}})
        

#        if all(e in v.keys() for e in rdcodeLinearisationIndices) \
#            and any(e for e in v["DisorderDisorderAssociation"] if e["DisorderDisorderAssociationType"] == parentPref[lang]):
        globals()[f"rdcode_orphalinearisation_{lang}"].update({ k : {x : y for (x, y) in v.items() if x in rdcodeLinearisationIndices}})

        tmp = {}
        tmp.update({ k : {x : y for (x, y) in v.items() if x in rdcodeNomenclatureIndices}})
        try:
            tmp[k]["DisorderDisorderAssociation"] = \
            [x for x in tmp[k]["DisorderDisorderAssociation"] if x["DisorderDisorderAssociationType"] != "Preferential parent"]
        except:
            tmp[k]["DisorderDisorderAssociation"] = []
        for l,w in tmp.items():
            if "ORPHAcode" in w:
                globals()[f"rdcode_orphanomenclature_{lang}"][l] = w
            
        #classif
        for classif in classifList:
            #if any(e in v.keys() for e in rdcodeClassifIndices)
            if "Classification" in v and classif in [x["ID of the classification"] for x in v["Classification"]]:
                globals()[f"rdcode_orphaclassification_{classif}_{lang}"].update({ k : {x : y for (x, y) in v.items() if x in rdcodeClassifIndices}})
                globals()[f"rdcode_orphaclassification_{classif}_{lang}"][k]["Classification"] = \
                    [x for x in globals()[f"rdcode_orphaclassification_{classif}_{lang}"][k]["Classification"] if x["ID of the classification"] == classif]
                inheritance = globals()[f"rdcode_orphaclassification_{classif}_{lang}"][k].pop("Inheritance") 
                globals()[f"rdcode_orphaclassification_{classif}_{lang}"][k].update({"Parent" : inheritance[classif]["Parent"]})
                globals()[f"rdcode_orphaclassification_{classif}_{lang}"][k].update({"Child" : inheritance[classif]["Children"]})

In [14]:
#print(rdcode_orpha_icd10_mapping_en["558"])
#print(rdcode_orphanomenclature_pt["558"])
print(len(packorpha_en_product))

11239


In [15]:
for var in list(globals()):
    if ('rdcode_' in var):
        local = len(globals()[var])
        if (local != 11239):
            print(var, local)

rdcode_orphaclassification_146_cs 227
rdcode_orphaclassification_147_cs 3791
rdcode_orphaclassification_148_cs 262
rdcode_orphaclassification_150_cs 1115
rdcode_orphaclassification_152_cs 279
rdcode_orphaclassification_156_cs 6905
rdcode_orphaclassification_181_cs 3339
rdcode_orphaclassification_182_cs 238
rdcode_orphaclassification_183_cs 228
rdcode_orphaclassification_184_cs 301
rdcode_orphaclassification_185_cs 231
rdcode_orphaclassification_186_cs 106
rdcode_orphaclassification_187_cs 1095
rdcode_orphaclassification_188_cs 511
rdcode_orphaclassification_189_cs 1237
rdcode_orphaclassification_193_cs 720
rdcode_orphaclassification_194_cs 611
rdcode_orphaclassification_195_cs 375
rdcode_orphaclassification_196_cs 377
rdcode_orphaclassification_197_cs 139
rdcode_orphaclassification_198_cs 318
rdcode_orphaclassification_199_cs 1011
rdcode_orphaclassification_200_cs 492
rdcode_orphaclassification_201_cs 171
rdcode_orphaclassification_202_cs 1095
rdcode_orphaclassification_203_cs 223
rdco

# Indices files creation

In [48]:
if not os.path.exists("./ElasticIndices/"):
    os.mkdir("./ElasticIndices/")
'''
if not os.path.exists("./ElasticIndices/packorpha/"):
    os.mkdir("./ElasticIndices/packorpha/")
if not os.path.exists("./ElasticIndices/rdcode/"):
    os.mkdir("./ElasticIndices/rdcode/")
'''
FileIndex = []

### rdcode_diff

index = f"rdcode_orpha_diff"
index_output = f"ElasticIndices/index_{index}.txt"
es_index = '{"index": {"_index":"' + index + '"}}\n'
with open(index_output, "w", encoding="utf-8") as file:
    for index, row in diff.iterrows():
        file.write(es_index)
        file.write(json.dumps(row.to_dict(), ensure_ascii=False))
        file.write('\n')
        

for lang in langs:

    ### packorpha_en_product
    if lang == "en":
        index = f"packorpha_en_product"
        index_output = f"ElasticIndices/index_{index}.txt"
        es_index = '{"index": {"_index":"' + index + '"}}\n'
        with open(index_output, "w", encoding="utf-8") as file:
            for local_dico in packorpha_en_product.values():
                file.write(es_index)
                file.write(json.dumps(local_dico, ensure_ascii=False))
                file.write('\n')
        

    ### rdcode indices
    '''
    if not os.path.exists(f"./ElasticIndices/rdcode/rdcode_{lang}/"):
        os.mkdir(f"./ElasticIndices/rdcode/rdcode_{lang}/")
    if not os.path.exists(f"./ElasticIndices/rdcode/rdcode_{lang}/Classifications/"):
        os.mkdir(f"./ElasticIndices/rdcode/rdcode_{lang}/Classifications/")
    '''
    ## indices definition

    FileIndex.extend([
    f"rdcode_orpha_icd10_mapping_{lang}",
    f"rdcode_orpha_icd11_mapping_{lang}",
    f"rdcode_orpha_omim_mapping_{lang}",
    f"rdcode_orpha_snomed_mapping_{lang}",
    f"rdcode_orphalinearisation_{lang}",
    f"rdcode_orphanomenclature_{lang}"
    ])
    
    for file in glob.glob(f"../../../NomenclaturePack25/Orphanet_Nomenclature_Pack_{lang.upper()}/Classifications/*"):
        basename = os.path.basename(file)
        FileIndex.append("rdcode_orphaclassification_" + basename.split("_")[1].split("_")[0] + f"_{lang}")

        ## iterates on indices to create them 
for index_output in FileIndex:
    lang = index_output.split("_")[-1]
    #sub_folder = "Classifications/" if "classification" in index_output else ""
    sub_folder = ""
    es_index = '{"index": {"_index":"' + index_output + '"}}\n'
    #with open(f"./ElasticIndices/rdcode_{lang}/{sub_folder}{index_output}.txt", "w", encoding="utf-8") as file:
    with open(f"./ElasticIndices/{index_output}.txt", "w", encoding="utf-8") as file:
        for _, local_dico in globals()[index_output].items():
            file.write(es_index)
            file.write(json.dumps(local_dico, ensure_ascii=False))
            file.write('\n')
        file.close()



# Injecting indices into Elasticsearch

In [31]:
"""
#packorpha
index_output = "packorpha_en_product"
index = "ElasticIndices/packorpha/index_packorpha_en_product.txt"
try:
    del_req = es_server.indices.delete(index=index_output)
    req = es_server.indices.create(index=index_output, body=body)
except Exception as e:
    print() 
new_array = []
with open(index, "r", encoding="utf-8") as index_final:
    for line in index_final.readlines():
        new_array.append(line)
    new_array.append('\n')
    try:
        res = es_server.bulk(body=''.join(new_array))
    except:
        print("Error on index: " + index)


#rdcode
for index_output in FileIndex:
    lang = index_output.split("_")[-1]
    sub_folder = "Classifications/" if "classification" in index_output else ""
    index = f"./ElasticIndices/rdcode/rdcode_{lang}/{sub_folder}{index_output}.txt"
    try:
        del_req = es_server.indices.delete(index=index_output)
        req = es_server.indices.create(index=index_output, body=body)
    except Exception as e:
        print(e) 
    new_array = []
    with open(index, "r", encoding="utf-8") as index_final:
        for line in index_final.readlines():
            new_array.append(line)
        new_array.append('\n')
        try:
            res = es_server.bulk(body=''.join(new_array))
        except Exception as e:
            print("Error on index: " + index)
            print(e)
    print("Successful injection for: " + index_output)

#rdcode diff
index = "ElasticIndices/index_rdcode_orpha_diff.txt"
index_output = "rdcode_orpha_diff"
try:
    del_req = es_server.indices.delete(index=index_output)
    req = es_server.indices.create(index=index_output, body=body)
except Exception as e:
    print(e) 
new_array = []
with open(index, "r", encoding="utf-8") as index_final:
    for line in index_final.readlines():
        new_array.append(line)
    new_array.append('\n')
    try:
        res = es_server.bulk(body=''.join(new_array))
    except Exception as e:
        print("Error on index: " + index)
        print(e)
"""

Successful injection for: rdcode_orphanomenclature_pt


'\n\n#rdcode diff\nindex = "ElasticIndices/index_rdcode_orpha_diff.txt"\nindex_output = "rdcode_orpha_diff"\ntry:\n    del_req = es_server.indices.delete(index=index_output)\n    req = es_server.indices.create(index=index_output, body=body)\nexcept Exception as e:\n    print(e) \nnew_array = []\nwith open(index, "r", encoding="utf-8") as index_final:\n    for line in index_final.readlines():\n        new_array.append(line)\n    new_array.append(\'\n\')\n    try:\n        res = es_server.bulk(body=\'\'.join(new_array))\n    except Exception as e:\n        print("Error on index: " + index)\n        print(e)\n'

# Download indices from Elasticsearch

In [56]:
'''
def exporter_index(client, index, fichier_dest):
    print(f"Début de l'extraction de l'index '{index}'...")
    compteur = 0
    try:
        # 2. Utilisation du helper 'scan' pour récupérer TOUS les documents
        # '_source' récupère le corps du document. Vous pouvez ajouter une query si besoin.
        requete_scan = scan(
            client=client,
            index=index,
            query={"query": {"match_all": {}}},
            _source=True 
        )

        # 3. Écriture des documents dans le fichier au format JSON Lines (un JSON par ligne)
        with open(fichier_dest, "w", encoding="utf-8") as f:
            for hit in requete_scan:
                # hit contient les métadonnées (_index, _id) et le contenu complet (_source)
                # On extrait généralement le contenu brut du document :
                document = hit["_source"]
                # Optionnel : Conserver l'ID Elasticsearch dans le document exporté
                document["_es_id"] = hit["_id"]

                f.write(json.dumps(document, ensure_ascii=False) + "\n")
                
                compteur += 1
            print(f"Succès ! {compteur} documents ont été écrits dans '{fichier_dest}'.")
    except Exception as e:
        print(f"Une erreur est survenue lors de l'export : {e}")


for file in FileIndex:
#for file in ['rdcode_orpha_diff']:
    OUTPUT_FILE ="ES_Indices_PROD_Pack_25/" + file + ".json"
    exporter_index(es_server_prod, index, OUTPUT_FILE)
'''

Début de l'extraction de l'index 'packorpha_en_product'...
Succès ! 11239 documents ont été écrits dans 'ES_Indices_PROD_Pack_25/rdcode_orpha_diff.json'.


# Debugging

In [9]:
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as file:
        detector = chardet.universaldetector.UniversalDetector()
        for line in file:
            detector.feed(line)
            if detector.done:
                break
        detector.close()
    return detector.result['encoding']

detect_encoding("../NomenclaturePack/Orphanet_Nomenclature_Pack_EN/Classifications/ORPHAclassification_181_rare_neurological_diseases_en_2025.xml")

'utf-8'

### Actual values

In [8]:
for lang in langs:
    print(f"--- LANGUE: \n{lang}\n---")
    print("ICD10:\n", len(globals()[f"rdcode_orpha_icd10_mapping_{lang}"]))
    print("ICD11:\n", len(globals()[f"rdcode_orpha_icd11_mapping_{lang}"]))
    print("OMIM:\n", len(globals()[f"rdcode_orpha_omim_mapping_{lang}"]))
    print("SNOMED:\n", len(globals()[f"rdcode_orpha_snomed_mapping_{lang}"]))
    print("Linearisation:\n", len(globals()[f"rdcode_orphalinearisation_{lang}"]))
    print("Nomenclature:\n", len(globals()[f"rdcode_orphanomenclature_{lang}"]))
    print(f"-- CLASSIFS:")
    for var in [x for x in globals() if "classification" in x and f"_{lang}" in x]:
        print(var, len(globals()[var]))

--- LANGUE: 
cs
---
ICD10:
 7534
ICD11:
 5987
OMIM:
 4940
SNOMED:
 6773
Linearisation:
 7610
Nomenclature:
 11239
-- CLASSIFS:
rdcode_orphaclassification_146_cs 227
rdcode_orphaclassification_147_cs 3791
rdcode_orphaclassification_148_cs 262
rdcode_orphaclassification_150_cs 1115
rdcode_orphaclassification_152_cs 279
rdcode_orphaclassification_156_cs 6905
rdcode_orphaclassification_181_cs 3339
rdcode_orphaclassification_182_cs 238
rdcode_orphaclassification_183_cs 228
rdcode_orphaclassification_184_cs 301
rdcode_orphaclassification_185_cs 231
rdcode_orphaclassification_186_cs 106
rdcode_orphaclassification_187_cs 1095
rdcode_orphaclassification_188_cs 511
rdcode_orphaclassification_189_cs 1237
rdcode_orphaclassification_193_cs 720
rdcode_orphaclassification_194_cs 611
rdcode_orphaclassification_195_cs 375
rdcode_orphaclassification_196_cs 377
rdcode_orphaclassification_197_cs 139
rdcode_orphaclassification_198_cs 318
rdcode_orphaclassification_199_cs 1011
rdcode_orphaclassification_200

In [ ]:
for k, v in rdcode_orphaclassification_199_en.items():
    print(k)

In [ ]:
import xml.etree.ElementTree as etree

tree = etree.parse("../NomenclaturePack/Orphanet_Nomenclature_Pack_EN/Classifications/ORPHAclassification_199_rare_bone_diseases_en_2025.xml")
root = tree.getroot()

sample_refs = tree.findall('.//OrphaCode')
for sample in sample_refs:
    print(sample.text)

### Expected Values

In [ ]:
for lang in langs:
    print(f"--- LANGUE: \n{lang}\n---")
    print("ICD10:") 
    os.system(f"grep -o '<Disorder ' NomenclaturePack/Orphanet_Nomenclature_Pack_{lang}/ORPHA_ICD10_mapping_{lang}_2025.xml | wc -l")
    print("ICD11:")
    os.system(f"grep -o '<Disorder ' NomenclaturePack/Orphanet_Nomenclature_Pack_{lang}/ORPHA_ICD11_mapping_{lang}_2025.xml | wc -l")
    print("OMIM:") 
    os.system(f"grep -o '<Disorder ' NomenclaturePack/Orphanet_Nomenclature_Pack_{lang}/ORPHA_OMIM_mapping_{lang}_2025.xml | wc -l")
    print("Linearisation:") 
    os.system(f"grep -o '<Disorder ' NomenclaturePack/Orphanet_Nomenclature_Pack_{lang}/ORPHAlinearisation_{lang}_2025.xml | wc -l")
    print("Nomenclature:") 
    os.system(f"grep -o '<Disorder ' NomenclaturePack/Orphanet_Nomenclature_Pack_{lang}/ORPHAnomenclature_{lang}_2025.xml | wc -l")
    print(f"-- CLASSIFS:")
    for var in [x for x in globals() if "classification" in x and f"_{lang}" in x]:
        nb = var.split("_")[2]
        print(var)
        os.system(f"grep -o '<Disorder ' NomenclaturePack/Orphanet_Nomenclature_Pack_en/Classifications/ORPHAclassification_{nb}* | wc -l")

In [ ]:
list(rdcode_orphaclassification_148_en.items())[-1]

In [ ]:
rdcode_orphaclassification_148_en["97965"]